# Batch-Hartigan failure-rate figures — low-dimensional study (`data/results_v2.csv.gz`)

Two figures (paper Section 5.1), each a **4 × 5 grid of panels**:

- **Rows** sweep the spread pair $\sigma^2_{\rm intra}/\sigma^2_{\rm inter}$ = 1/10, 4/7, 7/4, 10/1
  (separability ratio $\sigma^2_{\rm inter}/\sigma^2_{\rm intra}$ = 10, 1.75, 0.57, 0.1; top = most separable).
  Exploratory analyses (not part of this repository) showed the loop rate depends on the spreads **only through this ratio**, so one axis suffices.
- **Columns** sweep dimensionality d = 3, 6, 9, 12, 15.
- **Within each panel**: x = cluster size (19 irregular values 2…300, categorical spacing, labeled on the
  bottom-left panel only), y = amount of clusters K (2…15). **One pixel = one regime = 1000 trials.**

A trial ends in exactly one of four ways: clean convergence, an assignment **cycle** (`loop_count`),
an **empty cluster** (`zero_count`), or budget exhaustion (`max_iter_count` — fired once in 372.4M trials).

1. **Infinite-loop rate** = loop / trials → `figures/main_loop_rate.svg`
2. **Empty-cluster rate** = zero / trials → `figures/main_zero_rate.svg`

Both figures share a fixed 0–100% cividis scale, so they are pixel-for-pixel comparable.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns
import numpy as np
import pandas as pd
import pathlib

color1, color2 = "#1f77b4", "#fe6100"
sns.set_context("paper")
sns.set_style("white")
rcParams["font.family"] = "sans-serif"
fs1, fs2 = 28, 23
rcParams["font.size"] = rcParams["axes.labelsize"] = rcParams["axes.titlesize"] = fs1
rcParams["xtick.labelsize"] = rcParams["ytick.labelsize"] = fs1
rcParams["legend.fontsize"] = fs2

In [ ]:
df = pd.read_csv("data/results_v2.csv.gz")       # regime-level: 372,400 rows, 1000 trials each

# The three failure modes are mutually exclusive outcomes of a trial:
df["loop_rate"]  = 100 * df.loop_count / df.trials
df["zero_rate"]  = 100 * df.zero_count / df.trials

CS_VALS = sorted(df.cluster_size.unique())        # 19 values: 2..10, 15, 20, 30, 50..300
K_VALS  = sorted(df.amount_of_clusters.unique())  # 14 values: 2..15
DIMS        = [3, 6, 9, 12, 15]                   # panel COLUMNS (dim sweep, left -> right)
RATIO_STEPS = [(1, 10), (4, 7), (7, 4), (10, 1)]  # panel ROWS (intra, inter), top -> bottom

In [ ]:
def panel_matrix(metric, dim, intra, inter):
    """One (dim, intra, inter) slice as a (14, 19) matrix: rows = K asc, cols = cluster size asc."""
    sub = df[(df.dim == dim)
           & (df.intracluster_variance == intra)
           & (df.intercluster_variance == inter)]
    P = sub.pivot_table(index="amount_of_clusters", columns="cluster_size", values=metric)
    return P.loc[K_VALS, CS_VALS].values

In [ ]:
def plot_failure_grid(metric, cbar_label, fig_fname=None):
    fig, ax = plt.subplots(
        len(RATIO_STEPS), len(DIMS),                       # 4 rows x 5 cols of panels
        figsize=(5 * len(DIMS), 4 * len(RATIO_STEPS)),     # 5x4 in per panel, as in plot_metrics
        sharex=True, sharey=True, layout="compressed",
    )
    for i, (a, b) in enumerate(RATIO_STEPS):               # row = ratio step
        for j, d in enumerate(DIMS):                       # col = dimension
            im = ax[i, j].imshow(
                panel_matrix(metric, d, a, b),
                origin="lower", cmap="cividis",            # same map/orientation as the NMI plots
                vmin=0, vmax=100,                          # FIXED scale: comparable across figures
                aspect="auto",                             # let the 19x14 grid fill the 5x4 panel
            )

    for j, d in enumerate(DIMS):
        ax[0, j].set_title(f"d = {d}")                     # column headers, top row only
    # Cluster-size labels: ONLY on the bottom-left panel (cleanliness), rotated 90 so the 19
    # irregular values (2..300) cannot collide: slot width ~0.26 in vs ~0.19 in glyph height.
    ax[-1, 0].set_xticks(range(len(CS_VALS)), CS_VALS, rotation=90, fontsize=13)
    for j in range(1, len(DIMS)):                          # hide labels on the other bottom panels
        ax[-1, j].tick_params(labelbottom=False)           # (tick positions still shared)
    for i, (a, b) in enumerate(RATIO_STEPS):
        ax[i, 0].set_ylabel(rf"$\sigma^2$ {a}/{b}" + "\nK")  # row label: spread pair + axis name
        ax[i, 0].set_yticks(range(0, len(K_VALS), 2),      # every 2nd K to avoid crowding
                            K_VALS[::2], fontsize=14)
    ax[-1, 0].set_xlabel("Cluster size")                   # x-title under the labeled panel

    cbar = fig.colorbar(im, ax=ax, shrink=0.85)            # ONE shared colorbar (single 0-100 scale)
    cbar.set_label(cbar_label, rotation=270, labelpad=30)

    if fig_fname is not None:                              # save idiom from the ICML notebooks
        pathlib.Path(fig_fname).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(fig_fname, bbox_inches="tight", dpi=600)
    return fig, ax

# Infinite-loop rate

Cycles only (`loop_count / trials`).

In [ ]:
plot_failure_grid("loop_rate", "Infinite-loop rate (%)", "figures/main_loop_rate.svg");

# Empty-cluster rate

Empty-cluster terminations only (`zero_count / trials`) — the competing risk that dominates tiny clusters (cs=2) at low separability.

In [ ]:
plot_failure_grid("zero_rate", "Empty-cluster rate (%)", "figures/main_zero_rate.svg");